# Part 3 · File IO
> open / csv / json / pickle / gzip / io.BytesIO / io.StringIO

## 1. open — 文件读写

In [ ]:
# 模式速查
# 'r'   读（默认），文本模式
# 'w'   写（覆盖），文本模式
# 'a'   追加
# 'x'   独占创建（文件已存在则 FileExistsError）
# 'b'   二进制模式，配合上面用：'rb','wb','ab'
# 'r+'  读写（不清空），文本

# --- 读文件 ---
with open('data.txt', 'r', encoding='utf-8') as f:
    content = f.read()          # 全部读入字符串
    lines   = f.readlines()     # 所有行，返回列表（含换行符）
    line    = f.readline()      # 读一行
    # 大文件：逐行迭代（省内存）
    for line in f:
        process(line.rstrip('\n'))

# --- 写文件 ---
with open('out.txt', 'w', encoding='utf-8') as f:
    f.write('hello\n')          # 写字符串（不自动加换行）
    f.writelines(['a\n','b\n']) # 批量写
    print('hello', file=f)      # print 重定向到文件

# --- 读写位置 ---
f.tell()            # 当前位置（字节数）
f.seek(0)           # 移到开头
f.seek(0, 2)        # 移到末尾（0=SEEK_SET,1=SEEK_CUR,2=SEEK_END）

# --- 二进制 ---
with open('img.png', 'rb') as f:
    data = f.read()    # bytes 对象
with open('copy.png', 'wb') as f:
    f.write(data)

## 2. csv — CSV 读写

In [ ]:
import csv

# --- 读 ---
# reader：每行是列表
with open('data.csv', newline='', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter=',', quotechar='"')
    header = next(reader)           # 跳过表头
    for row in reader:
        print(row)                  # row 是 list[str]

# DictReader：每行是字典（key=列名）
with open('data.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row['name'], row['age'])  # dict 访问

# --- 写 ---
rows = [['Alice', 30], ['Bob', 25]]
with open('out.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['name', 'age'])    # 写表头
    writer.writerows(rows)              # 批量写

# DictWriter：按字典写
fields = ['name', 'age']
data   = [{'name':'Alice','age':30}, {'name':'Bob','age':25}]
with open('out.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()    # 写列名行
    writer.writerows(data)

# --- 常用参数 ---
csv.reader(f, delimiter='\t')          # Tab 分隔
csv.reader(f, delimiter=';')           # 分号分隔（欧洲常见）
csv.writer(f, quoting=csv.QUOTE_ALL)   # 所有字段加引号

## 3. json — JSON 读写

In [ ]:
import json

# --- 字符串 ↔ Python 对象 ---
# json.loads → 字符串解析为 Python 对象
obj = json.loads('{"name":"Alice","age":30}')  # {'name':'Alice','age':30}

# json.dumps → Python 对象序列化为字符串
s = json.dumps({'name':'Alice','age':30})
s = json.dumps(obj, indent=2)                # 格式化输出（缩进2）
s = json.dumps(obj, ensure_ascii=False)      # 保留中文（不转义 Unicode）
s = json.dumps(obj, sort_keys=True)          # key 排序
s = json.dumps(obj, default=str)             # 自定义序列化（如 datetime → str）

# --- 文件 ---
with open('data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)         # 从文件读

with open('out.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

# --- JSONL（每行一个 JSON，大数据常用）---
# 写
with open('data.jsonl', 'w') as f:
    for record in records:
        f.write(json.dumps(record) + '\n')

# 读
records = []
with open('data.jsonl') as f:
    for line in f:
        records.append(json.loads(line.strip()))

# --- 自定义序列化 ---
from datetime import datetime, date
import decimal

def json_serial(obj):
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    if isinstance(obj, decimal.Decimal):
        return float(obj)
    raise TypeError(f'Type {type(obj)} not serializable')

json.dumps({'dt': datetime.now()}, default=json_serial)

## 4. pickle — Python 对象序列化

In [ ]:
import pickle

# 保存任意 Python 对象（模型、复杂数据结构）
obj = {'key': [1,2,3], 'func': lambda x: x}  # lambda 无法 pickle

with open('data.pkl', 'wb') as f:
    pickle.dump(obj, f)

with open('data.pkl', 'rb') as f:
    obj_loaded = pickle.load(f)

# bytes 形式（不写文件）
b = pickle.dumps(obj)
obj = pickle.loads(b)

# ⚠️ 安全警告：不要 pickle.load 来自不可信来源的数据（可执行任意代码）
# ⚠️ 不同 Python 版本的 pickle 文件可能不兼容

## 5. gzip / bz2 / lzma — 压缩文件

In [ ]:
import gzip, bz2, lzma

# gzip（最常用，gz 格式）
with gzip.open('data.csv.gz', 'rt', encoding='utf-8') as f:
    content = f.read()   # 透明解压，像普通文件一样读

with gzip.open('out.csv.gz', 'wt', encoding='utf-8') as f:
    f.write(content)

# 压缩/解压 bytes
compressed = gzip.compress(b'hello world' * 1000)
original   = gzip.decompress(compressed)

# pandas 直接读 gz
import pandas as pd
df = pd.read_csv('data.csv.gz', compression='gzip')
df.to_csv('out.csv.gz', compression='gzip', index=False)

# zipfile
import zipfile
with zipfile.ZipFile('archive.zip', 'r') as zf:
    zf.namelist()              # 列出文件
    zf.extract('data.csv')     # 解压单个文件
    zf.extractall('output/')   # 全部解压
    with zf.open('data.csv') as f:  # 直接读 zip 内文件
        df = pd.read_csv(f)

## 6. io — 内存文件

In [ ]:
import io
import pandas as pd

# io.StringIO  → 内存中的文本文件（API 测试、避免写磁盘）
text = 'name,age\nAlice,30\nBob,25'
df = pd.read_csv(io.StringIO(text))

# 将 DataFrame 转为字符串（不写文件）
buf = io.StringIO()
df.to_csv(buf, index=False)
csv_string = buf.getvalue()

# io.BytesIO  → 内存中的二进制文件
# 场景：从 S3/GCS 下载到内存，直接解析
import boto3
s3 = boto3.client('s3')
buf = io.BytesIO()
s3.download_fileobj('bucket', 'data.parquet', buf)
buf.seek(0)
df = pd.read_parquet(buf)

# 将 DataFrame 保存到内存 bytes（用于上传）
buf = io.BytesIO()
df.to_parquet(buf, index=False)
buf.seek(0)
s3.upload_fileobj(buf, 'bucket', 'out.parquet')

# io.TextIOWrapper  → 把 BytesIO 包装成文本
raw = io.BytesIO(b'hello\nworld')
text_io = io.TextIOWrapper(raw, encoding='utf-8')
for line in text_io:
    print(line.strip())

## 7. yaml / toml / dotenv

In [ ]:
# --- PyYAML ---
import yaml

with open('config.yaml') as f:
    config = yaml.safe_load(f)      # safe_load 比 load 更安全

with open('out.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

# 字符串形式
yaml_str = 'name: Alice\nage: 30'
obj = yaml.safe_load(yaml_str)  # {'name':'Alice','age':30}

# --- python-dotenv（加载 .env 文件）---
from dotenv import load_dotenv
import os
load_dotenv()                   # 加载当前目录 .env 到环境变量
load_dotenv('.env.prod')        # 指定文件
DB_URL = os.environ['DB_URL']  # 读取

# --- configparser（读 .ini / .cfg）---
import configparser
cfg = configparser.ConfigParser()
cfg.read('config.ini')
cfg['database']['host']         # 读取配置
cfg.get('database', 'port', fallback='5432')  # 有默认值